In [1]:
import keras
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping
import numpy as np
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
DATA_PATH = r'C:\Users\Cloud\Downloads\MEDIAPIPE\training_data'
actions = np.array(["name"])

label_map = {label: num for num, label in enumerate(actions)}
sequences = []
labels = []

for action in actions:
    print(f"Loading: {action}")

    action_path = os.path.join(DATA_PATH, action)

    if not os.path.exists(action_path):
        print(f"Folder not found: {action_path}")
        continue

    video_folders = [
        folder for folder in os.listdir(action_path)
        if os.path.isdir(os.path.join(action_path, folder))
    ]

    for video_name in video_folders:
        video_path = os.path.join(action_path, video_name)
        frames = []

        frame_files = sorted(
            os.listdir(video_path),
            key=lambda x: int(os.path.splitext(x)[0])
        )

        for frame_file in frame_files[:30]:  # تحميل 30 إطار فقط
            frame_path = os.path.join(video_path, frame_file)
            res = np.load(frame_path)
            frames.append(res)

        if len(frames) == 30:  # تجاهل أي فيديو ناقص
            sequences.append(frames)
            labels.append(label_map[action])
        else:
            print(f"Skipped (not 30 frames): {video_path}")

X = np.array(sequences)
y = np.array(labels)

print("اكتمل تحميل البيانات")
print("X shape:", X.shape)
print("y shape:", y.shape)


Loading: name
Folder not found: C:\Users\Cloud\Downloads\MEDIAPIPE\training_data\name
اكتمل تحميل البيانات
X shape: (0,)
y shape: (0,)


In [14]:
X = np.array(sequences)
y = to_categorical(labels).astype(int)

In [ ]:
X.shape

In [ ]:
y.shape

In [16]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.1)

In [20]:
model = keras.Sequential()
model.add(keras.layers.LSTM(128,return_sequences=True,activation='relu',input_shape=(30,150)))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.LSTM(64,return_sequences=True,activation='relu'))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.LSTM(64,return_sequences=False,activation='relu'))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.Dense(32, activation='relu'))
model.add(keras.layers.Dense(32, activation='relu'))
model.add(keras.layers.Dense(actions.shape[0], activation='softmax'))


In [21]:
model.compile(optimizer='Adam',loss='categorical_crossentropy', metrics=['categorical_accuracy'])

In [22]:
early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model.fit(x_train, y_train,epochs=100,batch_size=32,callbacks=[early_stop],verbose=1)

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - categorical_accuracy: 0.3109 - loss: 1.0776
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - categorical_accuracy: 0.5718 - loss: 1.1286
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - categorical_accuracy: 0.5593 - loss: 1.8663
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - categorical_accuracy: 0.5266 - loss: 0.9807
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - categorical_accuracy: 0.6021 - loss: 1.8489
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - categorical_accuracy: 0.5674 - loss: 2.4123
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - categorical_accuracy: 0.6193 - loss: 3.0661
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - categorical_accuracy: 0.6029 - loss: 1.3597
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - categorical_accuracy: 0.5624 - loss: 2.4336
Epoch 10/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - categorical_accuracy: 0.5435 - loss: 2.2683
Epoch 11/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0

In [23]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Loss: {test_loss:.2f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 868ms/step - categorical_accuracy: 0.8333 - loss: 2.4078
Test Loss: 2.41
Test Accuracy: 83.33%


In [24]:
model.save(f"test model 2 {test_accuracy:.2f}.h5")